# 1. Recortar las fresas con Grounding DINO + SAM

Este cuaderno pasa todas las fotos de entrenamiento por Grounding DINO y SAM, igual que lo hará la página, y guarda cada fresa recortada de dos formas:

- **caja**: el recuadro de la fresa con un poco de margen, con el fondo que tenga.
- **negro**: el mismo recuadro pero con todo lo que no es fresa en negro, usando la máscara de SAM (hipótesis 4).

**Antes de empezar**

1. En *Entorno de ejecución → Cambiar tipo de entorno* escoger **GPU T4**.
2. En Google Drive, dentro de *Mi unidad*, crear una carpeta `fresas` y subir ahí `fresas_datos.zip` y `fotos_entrenamiento.zip`, que están en `frutas/datos/` del proyecto.

**Qué sale**

Al final queda `recortes.zip` en la misma carpeta `fresas` de Drive. Adentro están los recortes, `recortes.csv` con la foto y el grupo de cada recorte, `tiempos.csv` y la carpeta `revision` con nuestras fotos y el número de cada fresa encontrada.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CARPETA = "/content/drive/MyDrive/fresas"
DATOS = "/content/datos"
!mkdir -p $DATOS
!unzip -q -o "$CARPETA/fresas_datos.zip" -d $DATOS
!unzip -q -o "$CARPETA/fotos_entrenamiento.zip" -d $DATOS
!ls $DATOS

## Cargar los modelos
Son los mismos de la primera prueba. La primera vez se demora un poco porque los descarga.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor

disp = "cuda" if torch.cuda.is_available() else "cpu"
dino_proc = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
dino = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny").to(disp).eval()
sam_proc = SamProcessor.from_pretrained("facebook/sam-vit-base")
sam = SamModel.from_pretrained("facebook/sam-vit-base").to(disp).eval()
print("Listo en", disp)

## Funciones

- `detectar`: Grounding DINO con el texto `"a red strawberry."` y umbral 0.3, que fue lo que mejor funcionó en la primera prueba.
- `limpiar_cajas`: en las fotos de la Universidad Militar hay una sola fresa, así que se queda la caja con mayor puntaje. En las demás se quitan las cajas repetidas, las que encierran un grupo de fresas y los pedazos que quedan dentro de otra caja.
- `segmentar`: SAM saca la máscara de cada caja.
- `recortar`: guarda la versión con caja y la versión con fondo negro.
- `listar_fotos`: le pone a cada foto un **grupo** para dividir después sin fuga. En la Universidad Militar es un bloque de 50 fotos seguidas, en Jahangirnagar un bloque de 20 y en nuestras fotos la escena.

In [ ]:
import csv, time
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageDraw

TEXTO = "a red strawberry."  # el que mejor funcionó en la primera prueba
UMBRAL = 0.3
MARGEN = 0.10   # se agranda la caja un 10 % para no cortar el borde de la fresa
LADO = 256      # lado más largo del recorte guardado


def detectar(img):
    """Grounding DINO: cajas y puntajes de las fresas."""
    ent = dino_proc(images=img, text=TEXTO, return_tensors="pt").to(disp)
    with torch.no_grad():
        sal = dino(**ent)
    try:
        res = dino_proc.post_process_grounded_object_detection(
            sal, ent.input_ids, threshold=UMBRAL, text_threshold=UMBRAL, target_sizes=[img.size[::-1]])[0]
    except TypeError:  # versiones viejas de transformers
        res = dino_proc.post_process_grounded_object_detection(
            sal, ent.input_ids, box_threshold=UMBRAL, text_threshold=UMBRAL, target_sizes=[img.size[::-1]])[0]
    return res["boxes"].cpu().tolist(), res["scores"].cpu().tolist()


def segmentar(img, cajas):
    """SAM: una máscara por cada caja."""
    ent = sam_proc(img, input_boxes=[cajas], return_tensors="pt").to(disp)
    with torch.no_grad():
        sal = sam(**ent, multimask_output=False)
    return sam_proc.image_processor.post_process_masks(
        sal.pred_masks.cpu(), ent["original_sizes"].cpu(), ent["reshaped_input_sizes"].cpu())[0][:, 0].numpy()


def iou_cajas(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    area = lambda c: (c[2] - c[0]) * (c[3] - c[1])
    return inter / (area(a) + area(b) - inter + 1e-9)


def limpiar_cajas(cajas, puntajes, una_sola):
    """Devuelve los índices de las cajas que se quedan.
    Si la foto tiene una sola fresa se queda la de mayor puntaje. Si tiene varias se quitan
    las cajas repetidas, las cajas grandes que encierran a dos o más fresas a la vez
    y los pedazos que quedan casi enteros dentro de otra caja."""
    orden = sorted(range(len(cajas)), key=lambda i: -puntajes[i])
    if una_sola:
        return orden[:1]
    def dentro(i, j):  # qué parte de la caja i queda dentro de la caja j
        a, b = cajas[i], cajas[j]
        ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
        return ix * iy / ((a[2] - a[0]) * (a[3] - a[1]) + 1e-9)
    def centro_en(j, i):  # el centro de la caja j cae dentro de la caja i
        x0, y0, x1, y1 = cajas[i]
        return x0 <= (cajas[j][0] + cajas[j][2]) / 2 <= x1 and y0 <= (cajas[j][1] + cajas[j][3]) / 2 <= y1
    quedan = []
    for i in orden:
        if all(iou_cajas(cajas[i], cajas[j]) < 0.5 for j in quedan):
            quedan.append(i)
    quedan = [i for i in quedan if sum(centro_en(j, i) for j in quedan if j != i) < 2]
    return [i for k, i in enumerate(quedan) if all(dentro(i, j) < 0.8 for j in quedan[:k])]


def recortar(arr, caja, mascara):
    """Dos recortes de la misma fresa: con la caja tal cual y con todo lo que no es fresa en negro."""
    h, w = mascara.shape
    x0, y0, x1, y1 = caja
    mx, my = (x1 - x0) * MARGEN, (y1 - y0) * MARGEN
    x0, y0 = max(0, int(x0 - mx)), max(0, int(y0 - my))
    x1, y1 = min(w, int(x1 + mx) + 1), min(h, int(y1 + my) + 1)
    con_caja = arr[y0:y1, x0:x1]
    con_negro = con_caja * mascara[y0:y1, x0:x1, None]
    salida = []
    for a in (con_caja, con_negro):
        im = Image.fromarray(a.astype(np.uint8))
        im.thumbnail((LADO, LADO))
        salida.append(im)
    return salida


def listar_fotos(datos):
    """(ruta, fuente, clase, grupo, una_sola) de cada foto. El grupo sirve después para dividir sin fuga."""
    fotos = []
    for p in sorted(Path(datos, "fresas_datos").glob("*/*/*.jpg")):
        fuente, clase, n = p.parts[-3], p.parts[-2], int(p.stem)
        bloque = n // 50 if fuente == "umng" else n // 20  # fotos seguidas son de las mismas fresas
        fotos.append((p, fuente, clase, f"{fuente}_{clase}_{bloque:02d}", fuente == "umng"))
    for p in sorted(Path(datos, "entrenamiento").glob("*/*.jpg")):
        fotos.append((p, "propias", "sin_etiqueta", p.parent.name, False))
    return fotos


def procesar(fotos, salida):
    salida = Path(salida)
    (salida / "revision").mkdir(parents=True, exist_ok=True)
    filas, tiempos, sin_fresa = [], [], []
    for k, (ruta, fuente, clase, grupo, una_sola) in enumerate(fotos, 1):
        img = Image.open(ruta).convert("RGB")
        t0 = time.time()
        cajas, puntajes = detectar(img)
        if not cajas:
            sin_fresa.append(str(ruta)); tiempos.append((fuente, time.time() - t0, 0)); continue
        idx = limpiar_cajas(cajas, puntajes, una_sola)
        cajas, puntajes = [cajas[i] for i in idx], [puntajes[i] for i in idx]
        mascaras = segmentar(img, cajas)
        tiempos.append((fuente, time.time() - t0, len(cajas)))
        arr = np.array(img)
        for j, (c, p, m) in enumerate(zip(cajas, puntajes, mascaras), 1):
            nombre = f"{ruta.stem}_{j:02d}.jpg"
            for variante, im in zip(("caja", "negro"), recortar(arr, c, m)):
                destino = salida / variante / fuente / clase / nombre
                destino.parent.mkdir(parents=True, exist_ok=True)
                im.save(destino, quality=92)
            filas.append({"recorte": f"{fuente}/{clase}/{nombre}", "fuente": fuente, "clase": clase, "grupo": grupo,
                          "foto": ruta.name, "puntaje": round(p, 3), "caja": [round(v) for v in c]})
        if fuente == "propias":  # foto con los números de cada recorte, para revisar y etiquetar
            dib = img.copy(); d = ImageDraw.Draw(dib)
            for j, c in enumerate(cajas, 1):
                d.rectangle(c, outline=(0, 255, 0), width=4)
                d.text((c[0] + 6, c[1] + 4), str(j), fill=(255, 255, 0), stroke_width=3, stroke_fill=(0, 0, 0),
                       font_size=max(24, img.width // 40))
            dib.thumbnail((1600, 1600)); dib.save(salida / "revision" / ruta.name, quality=88)
        if k % 100 == 0 or k == len(fotos):
            print(f"{k}/{len(fotos)} fotos, {len(filas)} recortes")
    with open(salida / "recortes.csv", "w", newline="", encoding="utf8") as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0])); w.writeheader(); w.writerows(filas)
    with open(salida / "tiempos.csv", "w", newline="", encoding="utf8") as f:
        w = csv.writer(f); w.writerow(["fuente", "segundos", "fresas"]); w.writerows(tiempos)
    (salida / "sin_fresa.txt").write_text("\n".join(sin_fresa), encoding="utf8")
    return filas, tiempos, sin_fresa

# autoprueba rápida del filtro de cajas
assert limpiar_cajas([[0,0,10,10],[1,1,10,10],[20,20,30,30]], [.9,.8,.7], False) == [0, 2]    # repetida fuera
assert limpiar_cajas([[0,0,100,100],[5,5,20,20],[50,50,70,70]], [.9,.8,.7], False) == [1, 2]  # caja de grupo fuera
assert limpiar_cajas([[0,0,50,50],[10,10,20,20],[60,60,90,90]], [.9,.8,.7], False) == [0, 2]  # pedazo dentro de otra fuera
print("Funciones listas")

## Recortar todo
Con GPU se demora más o menos media hora. Va mostrando el avance cada 100 fotos.

In [ ]:
fotos = listar_fotos(DATOS)
print(len(fotos), "fotos")
filas, tiempos, sin_fresa = procesar(fotos, "/content/recortes")

## Resumen y guardar en Drive

In [ ]:
import pandas as pd
r = pd.DataFrame(filas)
print("Recortes por fuente y clase:")
print(r.groupby(["fuente", "clase"]).size())
print()
print("Fotos donde no encontró ninguna fresa:", len(sin_fresa))
t = pd.DataFrame(tiempos, columns=["fuente", "segundos", "fresas"])
print()
print("Segundos por foto (DINO + SAM):")
print(t.groupby("fuente")["segundos"].describe()[["mean", "50%", "max"]].round(2))

!cd /content && zip -q -r "$CARPETA/recortes.zip" recortes
print("Guardado en Drive:", CARPETA + "/recortes.zip")